# 03 — Structured Output: Getting Typed Data from Agents

**What you'll learn:**
- How to define a Pydantic model as the agent's output schema
- How to get a typed Python object instead of free text
- Streaming with structured output

---

## Why Structured Output?

In insurance and banking, free-text responses aren't enough. You need to:
- **Parse claim reports** into database-ready records
- **Extract risk assessments** with numeric scores and factor lists
- **Feed agent outputs** into downstream systems (APIs, dashboards)

Structured output forces the LLM to return a JSON object matching your
Pydantic schema — and the framework parses it into a typed Python object for you.

In [1]:
import os

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

## Example 1: Parsing Insurance Claims

Define a `ClaimReport` model. The agent will extract structured fields
from a free-text claim narrative.

In [2]:
class ClaimReport(BaseModel):
    """Structured representation of an insurance claim."""

    claimant_name: str = Field(description="Full name of the person filing the claim")
    policy_number: str = Field(description="The insurance policy number")
    incident_date: str = Field(description="Date of the incident (YYYY-MM-DD)")
    incident_type: str = Field(description="Type of incident: auto_collision, property_damage, theft, natural_disaster, medical")
    estimated_loss: float = Field(description="Estimated monetary loss in USD")
    description: str = Field(description="Brief summary of what happened")

In [3]:
agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="ClaimsParser",
    instructions=(
        "You are a claims processing assistant. Extract structured claim information "
        "from the provided narrative. Be accurate and conservative with estimates."
    ),
)

In [4]:
claim_narrative = """
My name is Robert Martinez and I need to file a claim on my policy POL-44821.
On January 15th, 2026, I was rear-ended at a stoplight on Main Street by another
driver who wasn't paying attention. The rear bumper is completely crushed, the
trunk won't close, and the tail lights are shattered. The body shop estimated
repairs at about $6,800. No injuries, thankfully.
"""

result = await agent.run(
    claim_narrative,
    options={"response_format": ClaimReport},
)

# result.value is a typed ClaimReport instance
claim = result.value

print(f"Claimant:      {claim.claimant_name}")
print(f"Policy:        {claim.policy_number}")
print(f"Date:          {claim.incident_date}")
print(f"Type:          {claim.incident_type}")
print(f"Est. Loss:     ${claim.estimated_loss:,.2f}")
print(f"Description:   {claim.description}")

Claimant:      Robert Martinez
Policy:        POL-44821
Date:          2026-01-15
Type:          auto_collision
Est. Loss:     $6,800.00
Description:   Rear-ended at a stoplight on Main Street. Damage includes a crushed rear bumper, non-closing trunk, and shattered tail lights. Estimated repair costs are $6,800.


Notice that `result.value` is a real `ClaimReport` Python object — you get
autocomplete, type checking, and can pass it directly to your database layer.

## Example 2: Property Risk Assessment

A second structured output model — this time evaluating risk from a
property description.

In [5]:
class RiskAssessment(BaseModel):
    """Risk assessment output for property insurance."""

    risk_score: int = Field(description="Risk score from 1 (very low) to 10 (very high)")
    risk_factors: list[str] = Field(description="List of identified risk factors")
    recommendation: str = Field(description="One of: 'standard_rate', 'elevated_rate', 'requires_inspection', 'decline'")
    reasoning: str = Field(description="Brief explanation of the assessment")

In [6]:
risk_agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="RiskAssessor",
    instructions=(
        "You are a property insurance risk assessor. Evaluate the property description "
        "and provide a structured risk assessment. Consider location, construction, "
        "age, hazards, and environmental factors."
    ),
)

property_description = """
3-bedroom wood-frame house built in 1978, located in a coastal flood zone in
southeastern Florida. The roof was last replaced in 2015. The property has a
detached garage and an above-ground pool. The area has experienced two
hurricanes in the past 5 years. The nearest fire station is 12 miles away.
"""

result = await risk_agent.run(
    property_description,
    options={"response_format": RiskAssessment},
)

assessment = result.value

print(f"Risk Score:      {assessment.risk_score}/10")
print(f"Recommendation:  {assessment.recommendation}")
print("Risk Factors:")
for factor in assessment.risk_factors:
    print(f"  • {factor}")
print(f"Reasoning:       {assessment.reasoning}")

Risk Score:      8/10
Recommendation:  elevated_rate
Risk Factors:
  • Coastal flood zone location
  • Wood-frame construction
  • Exposure to hurricanes
  • Distance from fire station
Reasoning:       The property is located in an area prone to flooding and hurricane damage, constructed using materials susceptible to fire, and is relatively distant from emergency services, increasing the overall risk.


## Streaming with Structured Output

You can combine streaming (for real-time UX) with structured output.
Use `stream=True`, iterate for live updates, then call
`get_final_response()` to get the parsed Pydantic object.

In [7]:
another_narrative = """
Hi, this is Angela Wu, policy POL-55190. Last Tuesday, March 18, 2026, a pipe
burst in my basement and flooded the entire lower level. The water damaged the
drywall, carpet, and a lot of stored furniture. A contractor came out and said
it would be around $15,000 to repair everything and replace the ruined items.
"""

stream = agent.run(
    another_narrative,
    stream=True,
    options={"response_format": ClaimReport},
)

# Show raw tokens as they arrive
print("Streaming raw tokens:")
async for update in stream:
    if update.text:
        print(update.text, end="", flush=True)
print("\n")

# Then get the final parsed object
final = await stream.get_final_response()
claim2 = final.value

print("--- Parsed ClaimReport ---")
print(f"Claimant:    {claim2.claimant_name}")
print(f"Policy:      {claim2.policy_number}")
print(f"Type:        {claim2.incident_type}")
print(f"Est. Loss:   ${claim2.estimated_loss:,.2f}")

Streaming raw tokens:
{"claimant_name":"Angela Wu","policy_number":"POL-55190","incident_date":"2026-03-18","incident_type":"property_damage","estimated_loss":15000,"description":"A pipe burst in the basement, causing extensive water damage to the lower level of the property, affecting drywall, carpet, and stored furniture."}

--- Parsed ClaimReport ---
Claimant:    Angela Wu
Policy:      POL-55190
Type:        property_damage
Est. Loss:   $15,000.00


In [8]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- Define a Pydantic `BaseModel` → pass it as `options={"response_format": ...}` → get a typed `.value`
- The LLM is **constrained** to produce valid JSON matching your schema
- Streaming + structured output work together via `ResponseStream.get_final_response()`
- This is essential for any pipeline where agent output feeds into downstream systems

**Next up:** [04 — Multi-Turn Conversations](./04-multi-turn-conversations.ipynb) —
build a banking chatbot that remembers context across turns.